# Ingest data with Lakeflow Connect

## Understand ingestion pipelines

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/02-understand-ingestion-pipelines.svg)

In [ ]:
# Lakeflow Connect provides managed connectors for ingesting data from external sources into Unity Catalog tables.
# Core components of an ingestion pipeline:
# 1. Connection - stores credentials/endpoint for source system (reusable)
# 2. Ingestion definition - specifies tables/objects and destination catalog/schema
# 3. Pipeline schedule - controls frequency of data flow

# For database connectors (e.g. SQL Server), an ingestion gateway is created
# For SaaS connectors (e.g. Salesforce), the connector handles extraction directly
print("Lakeflow Connect pipeline components: Connection, Ingestion Definition, Pipeline Schedule")


## Configure ingestion pipelines

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/02-configure-ingestion-pipelines.svg)

In [ ]:
# Ingestion pipelines can be created via:
# - Databricks UI (wizard): navigate to Data Ingestion, select connector
# - Asset Bundles (YAML) for version-controlled, declarative configuration
# - Notebooks or CLI

# Example YAML configuration (Asset Bundle):
yaml_config = '''
resources:
  pipelines:
    pipeline_sqlserver:
      name: customer-orders-pipeline
      catalog: sales
      schema: bronze
      ingestion_definition:
        ingestion_gateway_id: ${resources.pipelines.gateway.id}
        objects:
          - table:
              source_schema: dbo
              source_table: orders
              destination_catalog: sales
              destination_schema: bronze
'''
print("YAML-based pipeline config loads orders -> sales.bronze.orders")


## Control data extraction behavior

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/02-control-data-extraction-behavior.svg)

In [ ]:
# Two extraction patterns:
# 1. Snapshot: full reload of source data
# 2. Incremental: CDC-based, captures only changes (INSERT/UPDATE/DELETE)

# SCD Type setting controls how destination table handles changes:
# SCD Type 1 - overwrites existing records (current state only)
# SCD Type 2 - preserves historical versions (__START_AT / __END_AT columns)

scd_yaml = '''
table_configuration:
  scd_type: SCD_TYPE_2
  sequence_by: last_modified
'''
print("SCD Type 2 preserves history with __START_AT and __END_AT columns")
print("SCD Type 1 always reflects current state (overwrites)")


## Optimize column selection and refresh strategies

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/02-optimize-column-selection.svg)

In [ ]:
# Column selection reduces data transfer and storage costs

include_yaml = '''
table_configuration:
  include_columns:
    - customer_id
    - order_date
    - total_amount
'''

# include_columns: only these columns ingested; new source columns NOT auto-added
# exclude_columns: all but these columns ingested; new source columns included by default

# Full Refresh: clears destination and reloads all records
# - Useful when schema changes or data inconsistencies occur
# - Can be scheduled via full_refresh_window to avoid peak hours
print("include_columns vs exclude_columns: different behavior for new columns!")


## Scale with multiple destinations

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/02-scale-multiple-destinations.svg)

In [ ]:
# A single pipeline can write to multiple destination catalogs/schemas
# Useful when different teams need isolated copies of source data

multi_dest_yaml = '''
objects:
  - table:
      source_table: customers
      destination_catalog: sales
      destination_schema: analytics
  - table:
      source_table: customers
      destination_catalog: marketing
      destination_schema: campaigns
'''

# Custom destination_table: ingest same source table to multiple destinations in same schema
# Monitor via pipeline details page, configure failure notifications, adjust schedules
print("One source table -> multiple destination catalogs (sales + marketing)")


# Ingest data with notebooks

## Ingest batch data with DataFrames

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/03-ingest-batch-data-dataframes.svg)

In [ ]:
# Reading files with spark.read
# CSV example:
# df_csv = (spark.read
#     .format("csv")
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .load("/Volumes/catalog/schema/volume/data.csv"))

# JSON:
# df_json = spark.read.format("json").load("/path/to/data.json")

# Parquet (columnar, analytics-optimized):
# df_parquet = spark.read.format("parquet").load("/path/to/data.parquet")

# XML (requires rowTag):
# df_xml = spark.read.format("xml").option("rowTag", "record").load("/path/to/data.xml")

# JDBC (relational databases):
# df_jdbc = (spark.read
#     .format("jdbc")
#     .option("url", "jdbc:sqlserver://server.database.windows.net:1433;database=mydb")
#     .option("dbtable", "schema.tablename")
#     .option("user", dbutils.secrets.get(scope="jdbc", key="username"))
#     .option("password", dbutils.secrets.get(scope="jdbc", key="password"))
#     .load())

# Write to Unity Catalog (Delta format by default):
# df.write.mode("overwrite").saveAsTable("catalog.schema.table_name")
# Modes: overwrite, append, ignore, error (default)

print("Write modes: overwrite | append | ignore | error(default)")


## Ingest streaming data with Structured Streaming

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/03-ingest-streaming-data.svg)

In [ ]:
# Streaming from Kafka:
# df_stream = (spark.readStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", "broker:9092")
#     .option("subscribe", "topic-name")
#     .option("startingOffsets", "latest")
#     .load())

# Streaming from file source (explicit schema required - inferSchema not supported!)
from pyspark.sql.types import StructType, StringType, IntegerType, TimestampType

defined_schema = (StructType()
    .add('id', IntegerType())
    .add('name', StringType())
    .add('created_at', TimestampType()))

# df_file_stream = (spark.readStream
#     .format("csv")
#     .option("header", "true")
#     .schema(defined_schema)
#     .load("/Volumes/catalog/schema/volume/incoming/"))

# Write streaming data:
# (df_stream
#     .writeStream
#     .format("delta")
#     .outputMode("append")    # append | update | complete
#     .option("checkpointLocation", "/checkpoints/my-stream")
#     .toTable("catalog.schema.streaming_table"))

print("Output modes: append (new rows), update (changed rows), complete (all rows)")
print("Checkpoint location enables fault-tolerant restarts")


## Handle semi-structured data

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/03-handle-semi-structured-data.svg)

In [ ]:
# VARIANT data type for flexible semi-structured JSON storage
from pyspark.sql.functions import parse_json, variant_get, col

# df = spark.read.text('/path/to/data.json')
# df_variant = df.select(parse_json(col('value')).alias('data'))

# Query specific field with variant_get():
# df_variant.select(variant_get(col('data'), '$.customer.name', 'string')).display()

# Use when:
# - JSON schema varies between records
# - Schema evolves over time
# - API responses / event payloads with unknown structure

print("VARIANT type + parse_json() + variant_get() for flexible JSON ingestion")
print("Use $.field.subfield path syntax to extract nested values")


# Ingest data with SQL methods

## Create tables from queries with CTAS

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/04-create-tables-ctas.svg)

In [ ]:
# %sql
# CREATE TABLE AS SELECT (CTAS) - combines creation + data population
# Schema is automatically derived from query results
# Creates Delta table by default (ACID, time travel, optimized performance)

# CREATE TABLE sales.bronze.customers AS
# SELECT
#     customer_id,
#     UPPER(customer_name) AS customer_name,
#     email,
#     created_date
# FROM external_staging.raw_customers
# WHERE customer_status = 'active';

# Reading from files using read_files():
# CREATE TABLE catalog.schema.sales_data AS
# SELECT * FROM read_files(
#     '/Volumes/catalog/schema/volume/sales/*.parquet',
#     format => 'parquet'
# );

# IMPORTANT: Fails if table already exists
# IF NOT EXISTS: skips execution entirely (doesn't update)
print("CTAS: best for initial loads and one-time migrations")
print("WARNING: Fails if table already exists")


## Refresh tables with CREATE OR REPLACE TABLE

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/04-refresh-tables-create-or-replace.svg)

In [ ]:
# %sql
# CREATE OR REPLACE TABLE - full table replacement preserving metadata and permissions

# CREATE OR REPLACE TABLE catalog.schema.daily_metrics AS
# SELECT
#     report_date,
#     SUM(revenue) AS total_revenue,
#     COUNT(DISTINCT customer_id) AS unique_customers
# FROM catalog.schema.transactions
# WHERE report_date >= CURRENT_DATE - INTERVAL 30 DAYS
# GROUP BY report_date;

# Advantages over DROP + CREATE:
# - Table history preserved (Delta time travel)
# - Granted privileges retained
# - Row filters / column masks maintained
# - Downstream users unaffected

print("CREATE OR REPLACE: periodic full refresh while preserving permissions and history")


## Load files incrementally with COPY INTO

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/04-load-files-copy-into.svg)

In [ ]:
# %sql
# COPY INTO: idempotent file ingestion - already loaded files are skipped

# Target table must exist first!
# CREATE TABLE IF NOT EXISTS catalog.schema.events (
#     event_id STRING,
#     event_type STRING,
#     event_timestamp TIMESTAMP,
#     payload STRING
# );

# Basic COPY INTO:
# COPY INTO catalog.schema.events
# FROM '/Volumes/catalog/schema/volume/events/'
# FILEFORMAT = JSON
# FORMAT_OPTIONS ('multiline' = 'true');

# With file pattern selection:
# COPY INTO catalog.schema.orders
# FROM '/Volumes/catalog/schema/volume/orders/'
# FILEFORMAT = PARQUET
# PATTERN = 'orders_2024*.parquet';

# With schema evolution:
# COPY INTO catalog.schema.sensor_data
# FROM '/Volumes/catalog/schema/volume/sensors/'
# FILEFORMAT = CSV
# FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
# COPY_OPTIONS ('mergeSchema' = 'true');

# Validate without loading:
# COPY INTO catalog.schema.sensor_data FROM '/path/' FILEFORMAT = CSV VALIDATE ALL;

print("COPY INTO: idempotent - safe to re-run; already loaded files are skipped automatically")


## Choose the right method

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/04-choose-right-method.svg)

In [ ]:
# Decision guide for SQL ingestion methods:

decision = {
    "First time / one-time migration":         "CTAS",
    "Periodic full reload":                    "CREATE OR REPLACE TABLE",
    "Regular new file arrivals (idempotent)":  "COPY INTO",
    "Auto-detection + schema evolution":       "Auto Loader",
}
print('SQL Ingestion Method Decision Guide:')
for scenario, method in decision.items():
    print(f'  {scenario:45s} -> {method}')


# Ingest data with CDC feed

## Understand CDC ingestion patterns

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/05-understand-cdc-patterns.svg)

In [ ]:
# Change Data Capture (CDC): capture only INSERT/UPDATE/DELETE changes
# No full table reloads needed

# A CDC record contains:
# - Data columns: actual field values
# - Operation type: INSERT / UPDATE / DELETE
# - Sequence column: timestamp or sequence number for ordering

# Benefits vs full table reload:
benefits = {
    "Reduced latency":   "changes flow in minutes (vs hours)",
    "Lower costs":       "fewer records = less compute",
    "Minimal source impact": "reads change logs, not full table scans",
}
for b, desc in benefits.items():
    print(f'  {b}: {desc}')


## Process CDC with the AUTO CDC API

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/05-process-cdc-auto-cdc-api.svg)

In [ ]:
# AUTO CDC API in Lakeflow Spark Declarative Pipelines
# Handles: deduplication, out-of-order records, SCD management automatically

# from pyspark import pipelines as dp
# from pyspark.sql.functions import col, expr

# Step 1: Source view
# @dp.view
# def employees_cdf():
#     return spark.readStream.table("bronze.employee_changes")

# Step 2: Target table
# dp.create_streaming_table("employees")

# Step 3a: SCD Type 1 (current state only)
# dp.create_auto_cdc_flow(
#     target="employees",
#     source="employees_cdf",
#     keys=["employee_id"],
#     sequence_by=col("sequence_num"),
#     apply_as_deletes=expr("operation = 'DELETE'"),
#     except_column_list=["operation", "sequence_num"],
#     stored_as_scd_type=1
# )

# Step 3b: SCD Type 2 (full history with __START_AT / __END_AT columns)
# dp.create_auto_cdc_flow(
#     target="employees_history",
#     ...
#     stored_as_scd_type=2
# )

print("SCD Type 1: overwrites record (Seattle -> Portland replaces city value)")
print("SCD Type 2: Seattle record gets END_AT, new Portland record added")


## Configure CDC flows in SQL

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/05-configure-cdc-flows-sql.svg)

In [ ]:
# %sql
# SQL AUTO CDC syntax

# CREATE OR REFRESH STREAMING TABLE employees;

# CREATE FLOW employees_cdc_flow
# AS AUTO CDC INTO employees
# FROM stream(bronze.employee_changes)
# KEYS (employee_id)
# APPLY AS DELETE WHEN operation = 'DELETE'
# SEQUENCE BY sequence_num
# COLUMNS * EXCEPT (operation, sequence_num)
# STORED AS SCD TYPE 1;

# SCD Type 2 with selective history tracking:
# CREATE FLOW employees_history_flow
# AS AUTO CDC INTO employees_history
# FROM stream(bronze.employee_changes)
# KEYS (employee_id)
# SEQUENCE BY sequence_num
# COLUMNS * EXCEPT (operation, sequence_num)
# STORED AS SCD TYPE 2
# TRACK HISTORY ON * EXCEPT (last_login);

print("TRACK HISTORY ON * EXCEPT (last_login) -> selective history tracking")
print("last_login changes: update in place. name/dept changes: new history row")


## Handle out-of-order events

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/05-handle-out-of-order-events.svg)

In [ ]:
# Problem: distributed systems deliver events out of order
# AUTO CDC solution: compare sequence_by values for same key
# If newer update already applied, older late-arriving record is IGNORED

events = [
    {"employee_id": 101, "city": "Portland", "sequence_num": 200},  # arrives first
    {"employee_id": 101, "city": "Seattle",  "sequence_num": 100},  # arrives late
]

# AUTO CDC ignores Seattle record (seq=100) because Portland (seq=200) already applied
result = max(events, key=lambda x: x['sequence_num'])
print(f"Correct final state: {result['city']} (seq={result['sequence_num']})")

print("No manual merge logic, timestamp comparisons, or version tracking needed")
print("AUTO CDC handles all of this automatically via the sequence column")


# Ingest data with Spark Structured Streaming

## Understand the Structured Streaming model

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/06-understand-structured-streaming-model.svg)

In [ ]:
# Structured Streaming: data stream = continuously growing unbounded table
# Same DataFrame API as batch - use spark.readStream instead of spark.read

# Core 3-step pattern:
# 1. Configure source:      spark.readStream ...
# 2. Apply transformations: .select(), .filter(), .withColumn() etc.
# 3. Write to sink:         .writeStream + checkpointLocation

use_cases = ['IoT sensor readings', 'Clickstream data', 'Financial transactions', 'Log ingestion']
print('Structured Streaming use cases:')
for uc in use_cases:
    print(f'  - {uc}')

print("Checkpoints: track processed records; enable fault-tolerant restarts")


## Connect to Kafka and Event Hubs

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/06-connect-kafka-event-hubs.svg)

In [ ]:
# Apache Kafka:
# df_stream = (spark.readStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", "broker-server:9092")
#     .option("subscribe", "sensor-readings")
#     .option("startingOffsets", "latest")  # or "earliest"
#     .load())

# Azure Event Hubs via Kafka-compatible interface:
# connection_string = dbutils.secrets.get(scope='eventhubs', key='connection-string')
# kafka_options = {
#     "kafka.bootstrap.servers": f"{eh_namespace}.servicebus.windows.net:9093",
#     "subscribe": eh_name,
#     "kafka.sasl.mechanism": "PLAIN",
#     "kafka.security.protocol": "SASL_SSL",
#     ...
# }

# Native Event Hubs connector:
# events = spark.readStream.format('eventhubs').option('eventhubs.connectionString', cs).load()
# body column (not value!) contains the payload

# Parse Kafka message (binary value -> string -> JSON):
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StringType, DoubleType

schema = StructType().add('device_id', StringType()).add('temperature', DoubleType())
# parsed_df = (df_stream
#     .select(col('value').cast('string').alias('json_value'))
#     .select(from_json(col('json_value'), schema).alias('data'))
#     .select('data.*'))

print("Always cast binary value column to string before parsing JSON")
print("Store credentials in Databricks Secrets, never hardcode!")


## Configure checkpoint locations

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/06-configure-checkpoint-locations.svg)

In [ ]:
# Checkpoints store offset info for fault-tolerant restart
# EACH streaming query MUST have its own UNIQUE checkpoint path!

# checkpoint_path = '/checkpoints/sensor-ingestion'

# query = (parsed_df.writeStream
#     .format("delta")
#     .option("checkpointLocation", checkpoint_path)
#     .toTable("catalog.schema.sensor_readings"))

best_practices = [
    'Use Azure Data Lake Storage (durable, reliable)',
    'Each query has its OWN unique checkpoint path',
    'Avoid storage lifecycle policies that might delete checkpoint files',
    'Back up checkpoint directories for production workloads',
]
print('Checkpoint best practices:')
for bp in best_practices:
    print(f'  + {bp}')


## Control processing with triggers

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/06-control-processing-triggers.svg)

In [ ]:
# Triggers control when/how frequently the streaming engine processes data

# Default (no trigger): continuous; next batch starts when previous ends
# .trigger(processingTime='10 seconds'): fixed micro-batch interval
# .trigger(availableNow=True): process all unprocessed data, then terminate
# .trigger(once=True): DEPRECATED - use availableNow instead

trigger_guide = {
    "Continuous (24/7)":           "No trigger",
    "Fixed interval":              ".trigger(processingTime='10 seconds')",
    "Scheduled incremental batch": ".trigger(availableNow=True) + Databricks Jobs",
}
print('Trigger options:')
for pattern, trigger in trigger_guide.items():
    print(f'  {pattern:35s} -> {trigger}')


## Monitor streaming queries

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/06-monitor-streaming-queries.svg)

In [ ]:
# Monitoring active streaming queries

# query.isActive          -> True if running
# query.status            -> current state dict
# query.recentProgress    -> list of recent micro-batch metrics

# Block until stream ends:
# try:
#     query.awaitTermination()
# except Exception as e:
#     print(f'Stream failed: {e}')

metrics = {
    "inputRowsPerSecond":     "How fast data arrives",
    "processedRowsPerSecond": "How fast we process (should be >= input)",
    "batchDuration":          "Time per micro-batch",
    "numInputRows":           "Rows in last batch",
}
print('Key streaming metrics to monitor:')
for metric, desc in metrics.items():
    print(f'  {metric:30s} -> {desc}')

print("Spark UI -> Structured Streaming tab for visual monitoring")


# Ingest data with Auto Loader

## Understand how Auto Loader works

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/07-understand-auto-loader.svg)

In [ ]:
# Auto Loader: Structured Streaming source for cloud storage file ingestion
# - Automatically detects new files as they appear
# - Exactly-once guarantees per file
# - No manual state management needed

print('File Detection Modes:')
print()
print('1. Directory Listing Mode:')
print('   - Periodically lists input directory')
print('   - Simple, no extra configuration')
print('   - Less efficient for large-scale workloads')
print()
print('2. File Notification Mode (RECOMMENDED for production):')
print('   - Uses Azure Blob/ADLS event notifications')
print('   - Lower latency, fewer cloud API calls')
print('   - Enable via file events on Unity Catalog external location')


## Configure Auto Loader for file ingestion

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/07-configure-auto-loader.svg)

In [ ]:
# Auto Loader uses format('cloudFiles') with spark.readStream

# base_path = 'abfss://container@storage.dfs.core.windows.net/autoloader/orders'
# schema_path = f'{base_path}/schema'
# checkpoint_path = f'{base_path}/checkpoint'

# (spark.readStream
#   .format('cloudFiles')
#   .option('cloudFiles.format', 'json')   # json, csv, parquet, avro, orc, xml, text, binary
#   .option('cloudFiles.schemaLocation', schema_path)
#   .load('abfss://container@storage.dfs.core.windows.net/incoming/orders/')
#   .writeStream
#   .option('checkpointLocation', checkpoint_path)
#   .trigger(availableNow=True)
#   .toTable('sales.bronze.orders'))

# SQL syntax:
# SELECT * FROM read_files(
#   'abfss://container@storage.dfs.core.windows.net/incoming/orders/',
#   format => 'json',
#   schemaHints => 'order_id INT, amount DECIMAL(10,2)'
# )

# In Lakeflow Declarative Pipelines (STREAM enables Auto Loader):
# CREATE OR REFRESH STREAMING TABLE bronze_orders
# AS SELECT * FROM STREAM read_files('abfss://...', format => 'json')

print('Tip: Keep schema location and checkpoint location SEPARATE')
print('This allows resetting a stream without losing the inferred schema')


## Handle schema inference and evolution

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/07-handle-schema-inference-evolution.svg)

In [ ]:
# Schema inference: Auto Loader samples files to determine column names and types
# Default for JSON/CSV: all columns as STRING (safe, prevents type mismatches)

# Enable automatic type detection:
# .option('cloudFiles.inferColumnTypes', 'true')

evolution_modes = {
    'addNewColumns':    'Stream FAILS; new cols added to schema; restart continues',
    'rescue':           'Schema never evolves; new cols -> _rescued_data column',
    'failOnNewColumns': 'Stream fails; restart requires schema update',
    'none':             'New columns ignored; data not rescued',
}
print('Schema Evolution Modes (cloudFiles.schemaEvolutionMode):')
for mode, desc in evolution_modes.items():
    print(f'  {mode:20s} -> {desc}')

print()
print('_rescued_data: captures data not matching expected schema (no data loss!)')
print('Schema hints: .option("cloudFiles.schemaHints", "order_date DATE, amount DECIMAL(10,2)")')


## Monitor and optimize Auto Loader streams

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/07-monitor-optimize-auto-loader.svg)

In [ ]:
# Query state of discovered files:
# SELECT * FROM cloud_files_state('/path/to/checkpoint');
# Returns: file path, size, processing status, discovery timestamp

# Rate limiting (prevents oversized micro-batches):
# .option('cloudFiles.maxFilesPerTrigger', '1000')
# .option('cloudFiles.maxBytesPerTrigger', '1g')

# Streaming metrics:
# - numFilesOutstanding: files discovered but not yet processed
# - numBytesOutstanding: total bytes waiting to be processed

# Access source file metadata:
# (spark.readStream
#   .format('cloudFiles')
#   .option('cloudFiles.format', 'parquet')
#   .load(source_path)
#   .select('*', '_metadata.file_path', '_metadata.file_modification_time'))

metadata_fields = ['file_path', 'file_name', 'file_size', 'file_modification_time']
print('Available _metadata fields:')
for f in metadata_fields:
    print(f'  _metadata.{f}')

print()
print('IMPORTANT: Never apply lifecycle policies that delete checkpoint files!')


# Ingest data with Lakeflow Spark Declarative Pipelines

## Understand Lakeflow Spark Declarative Pipelines for ingestion

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/08-understand-lakeflow-declarative-pipelines.svg)

In [ ]:
# Lakeflow Spark Declarative Pipelines: declarative framework for batch + streaming pipelines
# Define WHAT you want; the pipeline handles HOW to execute

benefits = {
    'Automatic orchestration':   'Determines execution order; maximizes parallelism',
    'Transient failure handling': 'Retries at most granular level (individual Spark tasks)',
    'Exactly-once processing':   'Each input row processed only once',
    'Incremental processing':    'Only new/changed data processed per update',
}
print('Pipeline benefits for ingestion:')
for benefit, desc in benefits.items():
    print(f'  + {benefit:35s}: {desc}')

print()
print('Source files: place in transformations folder in Git-connected workspace')
print('Pipeline Editor: view execution as DAG with table dependencies')


## Create streaming tables for ingestion

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/08-create-streaming-tables.svg)

In [ ]:
# SQL: streaming table from cloud files (STREAM keyword enables Auto Loader)
# CREATE OR REFRESH STREAMING TABLE orders_bronze
# AS SELECT *
# FROM STREAM read_files(
#   'abfss://container@storage.dfs.core.windows.net/incoming/orders/',
#   format => 'json'
# )

# Python: @dp.table decorator
# from pyspark import pipelines as dp

# @dp.table
# def orders_bronze():
#     return (
#         spark.readStream.format('cloudFiles')
#             .option('cloudFiles.format', 'json')
#             .load('abfss://container@storage.dfs.core.windows.net/incoming/orders/')
#     )

# Pipeline manages: checkpoints, schema inference/evolution, exactly-once processing
# Default flow: created automatically with table definition

print("SQL: STREAM keyword + read_files() -> Auto Loader enabled automatically")
print("Pipeline handles checkpoints, schema, retries automatically")


## Configure flows for multiple sources

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/08-configure-flows-multiple-sources.svg)

In [ ]:
# Explicit flows: ingest from multiple sources into a single target table
# More flexible than UNION - add new sources without modifying existing flows

# SQL:
# CREATE OR REFRESH STREAMING TABLE orders_us;

# CREATE FLOW orders_west_flow
# AS INSERT INTO orders_us BY NAME
# SELECT * FROM STREAM(orders_west);

# CREATE FLOW orders_east_flow
# AS INSERT INTO orders_us BY NAME
# SELECT * FROM STREAM(orders_east);

# Python:
# from pyspark import pipelines as dp
# dp.create_streaming_table('orders_us')

# @dp.append_flow(target='orders_us')
# def orders_west_flow():
#     return spark.readStream.table('orders_west')

# @dp.append_flow(target='orders_us')
# def orders_east_flow():
#     return spark.readStream.table('orders_east')

print("WARNING: Flow names identify streaming checkpoints!")
print("Renaming a flow resets its checkpoint - starts processing from the beginning")
print("Choose flow names carefully from the start!")


## Ingest from message buses

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/08-ingest-from-message-buses.svg)

In [ ]:
# Supported: Apache Kafka, Azure Event Hubs, Amazon Kinesis, Google Pub/Sub

# SQL with read_kafka():
# CREATE OR REFRESH STREAMING TABLE events_raw
# AS SELECT *
# FROM STREAM read_kafka(
#   bootstrapServers => 'kafka_server:9092',
#   subscribe => 'events_topic'
# )

# Python: Azure Event Hubs via Kafka-compatible interface
# from pyspark import pipelines as dp
# EH_CONN_STR = dbutils.secrets.get(scope='eh-secrets', key='connection-string')
# KAFKA_OPTIONS = {
#     'kafka.bootstrap.servers': f'{EH_NAMESPACE}.servicebus.windows.net:9093',
#     'subscribe': EH_NAME,
#     'kafka.sasl.mechanism': 'PLAIN',
#     'kafka.security.protocol': 'SASL_SSL',
#     ...
# }
# @dp.table
# def events_bronze():
#     return spark.readStream.format('kafka').options(**KAFKA_OPTIONS).load()

print("Security: always store connection strings in Databricks Secrets")
print("Use pipeline parameters for non-sensitive config (namespace, topic names)")


## Handle schema inference and evolution

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.ingest-data-into-unity-catalog/08-handle-schema-inference-evolution.svg)

In [ ]:
# Schema inference in pipelines: from_json() with NULL schema

# SQL:
# CREATE STREAMING TABLE events_parsed AS
# SELECT
#     from_json(value, NULL,
#         map('schemaLocationKey', 'events_schema',
#             'schemaEvolutionMode', 'addNewColumns')) AS parsed
# FROM STREAM read_kafka(
#     bootstrapServers => 'kafka_server:9092',
#     subscribe => 'events_topic'
# )

evolution_modes = {
    'addNewColumns':    'Auto-add new columns (DEFAULT)',
    'rescue':           'New columns -> _rescued_data column',
    'failOnNewColumns': 'Pipeline fails when new columns detected',
    'none':             'New columns ignored; not rescued',
}
print('Pipeline Schema Evolution Modes:')
for mode, desc in evolution_modes.items():
    print(f'  {mode:20s} -> {desc}')

print()
print('Monitor pipeline via event log, Pipeline Editor (DAG), and failure notifications')
